In [1]:
import ollama
import os
from pathlib import Path
import re
from IPython.display import Markdown, display

In [2]:
# 链接 llama4:scout
client = ollama.Client(host='http://127.0.0.1:11434')

In [3]:
# ===== 参数 =====
QUESTIONS_DIR = Path("Questions")
# MODEL_NAME = "llama4:scout"
MODEL_NAME = "qwen3-coder-next"

prompt_path = Path("prompt2.txt")
system_prompt = prompt_path.read_text(encoding="utf-8")


def list_question_dirs(questions_dir: Path):
    return sorted(
        [p for p in questions_dir.iterdir() if p.is_dir() and not p.name.startswith(".")],
        key=lambda x: x.name
    )


def list_explain_txts(explain_dir: Path):
    txts = []

    for p in explain_dir.iterdir():
        if not (p.is_file() and p.suffix.lower() == ".txt"):
            continue

        parts = p.stem.split()

        # 文件名格式："{global_idx} {temperature}.txt"
        if len(parts) == 2 and parts[0].isdigit():
            txts.append((int(parts[0]), p))

    txts_sorted = sorted(txts, key=lambda x: x[0])
    return [p for _, p in txts_sorted]


def parse_idx_and_temp(txt_path: Path):
    parts = txt_path.stem.split()
    if len(parts) != 2 or not parts[0].isdigit():
        raise ValueError(f"非法文件名格式: {txt_path.name}")

    global_idx = int(parts[0])
    temperature = float(parts[1])
    return global_idx, temperature

def extract_python_code_from_llm_output(text: str) -> str:
    """
    从 LLM 输出中提取可直接保存为 .py 的代码。

    处理规则：
    1. 如果存在 ```python ... ``` 或 ``` ... ``` 代码块，优先提取第一个代码块内容
    2. 否则返回原文本
    3. 去掉首尾空白
    """
    text = text.strip()

    # 优先匹配 ```python ... ```
    m = re.search(r"```python\s*\n(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        return m.group(1).strip()

    # 再匹配普通 ``` ... ```
    m = re.search(r"```\s*\n(.*?)```", text, flags=re.DOTALL)
    if m:
        return m.group(1).strip()

    return text

dirs = list_question_dirs(QUESTIONS_DIR)

In [4]:
display(dirs)

[PosixPath('Questions/Easy B3666'),
 PosixPath('Questions/Easy P15288'),
 PosixPath('Questions/Easy P15457'),
 PosixPath('Questions/Easy P4306'),
 PosixPath('Questions/Easy P7714'),
 PosixPath('Questions/Hard P11658'),
 PosixPath('Questions/Hard P11823'),
 PosixPath('Questions/Hard P13901'),
 PosixPath('Questions/Hard P15082'),
 PosixPath('Questions/Hard P6845'),
 PosixPath('Questions/Mid P1407'),
 PosixPath('Questions/Mid P14989'),
 PosixPath('Questions/Mid P3007'),
 PosixPath('Questions/Mid P3167'),
 PosixPath('Questions/Mid P4092')]

In [ ]:
# 对所有题目遍历输出code
for d in dirs:
    explain_dir = d / "LLM Explains"
    code_dir = d / "LLM Codes"
    code_dir.mkdir(parents=True, exist_ok=True)

    if not explain_dir.exists():
        print(f"skip: {explain_dir} not found")
        continue

    explain_files = list_explain_txts(explain_dir)

    for explain_path in explain_files:
        try:
            global_idx, temperature = parse_idx_and_temp(explain_path)
            user_input = explain_path.read_text(encoding="utf-8")

            response = ollama.chat(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_input},
                ],
                options={
                    "temperature": temperature,
                },
            )

            raw_output_text = response["message"]["content"]
            output_text = extract_python_code_from_llm_output(raw_output_text)
            
            output_path = code_dir / f"{global_idx} {temperature}.py"
            output_path.write_text(output_text, encoding="utf-8")

            print(f"done: {d.name} -> {output_path.name}")

        except Exception as e:
            print(f"error: {d.name}, file={explain_path.name} -> {e}")

In [5]:
# 测试index0全部解释，每个输出一个code
d = dirs[0]

explain_dir = d / "LLM Explains"
code_dir = d / "LLM Codes"
code_dir.mkdir(parents=True, exist_ok=True)

if not explain_dir.exists():
    print(f"skip: {explain_dir} not found")
else:
    explain_files = list_explain_txts(explain_dir)

    for explain_path in explain_files:
        try:
            global_idx, temperature = parse_idx_and_temp(explain_path)
            user_input = explain_path.read_text(encoding="utf-8")

            response = ollama.chat(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_input},    # 只喂解释
                    # {"role": "user", "content": (d / "Question.txt").read_text(encoding="utf-8")},    # 只喂原题
                    # {"role": "user", "content": (d / "Question.txt").read_text(encoding="utf-8") + user_input},    # 原题+解释
                ],
                options={
                    "temperature": temperature,
                },
            )

            raw_output_text = response["message"]["content"]
            output_text = extract_python_code_from_llm_output(raw_output_text)
            
            output_path = code_dir / f"{global_idx} {temperature}.py"
            output_path.write_text(output_text, encoding="utf-8")

            print(f"done: {d.name} -> {output_path.name}")
            print(response["prompt_eval_count"] + response["eval_count"])

        except Exception as e:
            print(f"error: {d.name}, file={explain_path.name} -> {e}")

done: Easy B3666 -> 1 0.0.py
3325
done: Easy B3666 -> 2 0.0.py
3325
done: Easy B3666 -> 3 0.0.py
3325
done: Easy B3666 -> 4 0.0.py
3325
done: Easy B3666 -> 5 0.0.py
3325
done: Easy B3666 -> 6 0.0.py
3325
done: Easy B3666 -> 7 0.0.py
3325
done: Easy B3666 -> 8 0.0.py
3325
done: Easy B3666 -> 9 0.0.py
3325
done: Easy B3666 -> 10 0.0.py
3325
done: Easy B3666 -> 11 0.0.py
3325
done: Easy B3666 -> 12 0.0.py
3325
done: Easy B3666 -> 13 0.0.py
3325
done: Easy B3666 -> 14 0.0.py
3325
done: Easy B3666 -> 15 0.0.py
3325
done: Easy B3666 -> 16 0.0.py
3325
done: Easy B3666 -> 17 0.0.py
3325
done: Easy B3666 -> 18 0.0.py
3325
done: Easy B3666 -> 19 0.0.py
3325
done: Easy B3666 -> 20 0.0.py
3325
done: Easy B3666 -> 21 0.2.py
3432
done: Easy B3666 -> 22 0.2.py
4879
done: Easy B3666 -> 23 0.2.py
4776
done: Easy B3666 -> 24 0.2.py
3602
done: Easy B3666 -> 25 0.2.py
6099
done: Easy B3666 -> 26 0.2.py
5599
done: Easy B3666 -> 27 0.2.py
4417
done: Easy B3666 -> 28 0.2.py
4769
done: Easy B3666 -> 29 0.2.py